#### **Step 1: Import Required Libraries**


In [14]:
from pyspark.sql.functions import (col, lower, upper, when, expr, regexp_replace, to_timestamp, coalesce, isnan, count, trim, round, to_date,
 date_format, bround, lit, initcap,array_distinct, sort_array, transform, concat_ws, current_date, datediff, abs, udf, current_timestamp, row_number, split)
from pyspark.sql.window import Window
from pyspark.sql.types import FloatType, TimestampType, DateType, StringType
from delta.tables import DeltaTable
from datetime import datetime

StatementMeta(, 814c0d20-b57c-4f2a-a4db-167e55982b69, 16, Finished, Available, Finished)

#### **Step 2: Load Raw Data from Lakehouse (Bronze Layer)**

In [15]:
# Create path variable
raw_transaction_path = "Files/cdp_bronze/oracle/raw/kyc_documents"
df_raw_kyc_documents = spark.read.format("parquet").load(raw_transaction_path)
display(df_raw_kyc_documents.limit(10))

StatementMeta(, 814c0d20-b57c-4f2a-a4db-167e55982b69, 17, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 6ea50c80-1637-40e6-908f-46f162d2b3ed)

### **Column-by-Column Transformation Plan**
document_id
- Current: Already in consistent custom format (e.g., fee18459-ab07-47c1-b842-d76efafb9b61).
- Action: No transformation needed.
- Final: Keep as-is.

user_id
- Current: Clean and consistent (e.g., BB1000xxxx).
- Action: No transformation needed.
- Final: Keep as-is.

status
- Current: Clean and consistent (e.g., 2025-02-03).
- Action: No transformation needed.
- Final: Keep as-is.

#### **Step 3: Inspect the kyc_documents dataframe for data quality checks**

In [16]:
# Select specific columns and count nulls
null_counts = df_raw_kyc_documents.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in 
     ['document_id','user_id','document_type', 'upload_date', 'status', 'verified_by']
    ]
)

# Display results
null_counts.show()

StatementMeta(, 814c0d20-b57c-4f2a-a4db-167e55982b69, 18, Finished, Available, Finished)

+-----------+-------+-------------+-----------+------+-----------+
|document_id|user_id|document_type|upload_date|status|verified_by|
+-----------+-------+-------------+-----------+------+-----------+
|          0|      0|            0|          0| 13602|       7314|
+-----------+-------+-------------+-----------+------+-----------+



#### **Step 4: Data Transformation to document_type column**

Issues:
- Inconsistent prefixes: Some values include the "Scan_" prefix (e.g., Scan_Passport, Scan_Utility Bill).
- Casing inconsistencies: Mixed casing across entries (e.g., Driver License vs. driver license).
- Duplicate logical categories: Multiple representations of the same document type (e.g., Passport vs. Scan_Passport).
- Potential whitespace issues: Some entries may include leading/trailing spaces.

Transformations:
- Remove non-standard prefixes: Strip off "Scan_" from all entries to retain the base document type.
- Normalize casing: Convert all values to title case (e.g., passport → Passport) using initcap().
- Trim whitespaces: Use trim() to remove unnecessary leading/trailing whitespace.
- Preserve high-level categories: Keep standardized values such as Passport, National ID, Utility Bill, and Driver License.
- Handle unknown values (if any): Replace unrecognized or blank entries with NULL or a placeholder like 'Unknown' if needed for consistency.


In [17]:
# Check distinct values in document_type
df_raw_kyc_documents.select("document_type").distinct().show()

# Normalize and transform document_type values
df_cleaned_kyc_documents = df_raw_kyc_documents.withColumn(
    "document_type",
    when(lower(trim(col("document_type"))).isin("utility bill", "scan_utility bill"), "Utility Bill")
    .when(lower(trim(col("document_type"))).isin("scan_national id", "national id"), "National ID")
    .when(lower(trim(col("document_type"))).isin("scan_driver license", "driver license"), "Driver License")
    .when(lower(trim(col("document_type"))).isin("scan_passport", "passport"), "Passport")
    .otherwise(None)  # Set anything else to NULL
)

# Display distinct values after transformation
df_cleaned_kyc_documents.select("document_type").distinct().show()

StatementMeta(, 814c0d20-b57c-4f2a-a4db-167e55982b69, 19, Finished, Available, Finished)

+-------------------+
|      document_type|
+-------------------+
|       Utility Bill|
|           Passport|
|  Scan_Utility Bill|
|   Scan_National ID|
|        National ID|
|      Scan_Passport|
|     Driver License|
|Scan_Driver License|
+-------------------+

+--------------+
| document_type|
+--------------+
|  Utility Bill|
|      Passport|
|   National ID|
|Driver License|
+--------------+



#### **Step 4: Data Transformation to document_type column**

Issues
- Inconsistent casing – e.g., Verified, invalid, UNKNOWN.
- Presence of nulls or empty strings – indicating unclear or missing review outcomes.
- Ambiguous or system-generated entries – such as UNKNOWN, invalid.
- Non-standard values – not aligned with typical KYC document lifecycle statuses.

Transformations
Normalized casing and values:
- All entries were lowercased and trimmed to remove leading/trailing whitespace.
- Known values (verified, rejected, pending, invalid, unknown) were mapped to title case standardized values:
- verified → Verified
- rejected → Rejected
- pending → Pending
- invalid → Invalid
- unknown / blanks → Unknown

Handled missing or invalid entries
- Nulls, blanks (''), and any unknown strings defaulted to "Unknown" to avoid analytics errors or silent failures.

In [18]:
# Check distinct values in status
df_cleaned_kyc_documents.select("status").distinct().show()

# Clean and normalize status values
df_cleaned_kyc_documents = df_cleaned_kyc_documents.withColumn(
    "status",
    when(trim(lower(col("status"))).isin("verified"), "Verified")
    .when(trim(lower(col("status"))).isin("rejected"), "Rejected")
    .when(trim(lower(col("status"))).isin("pending"), "Pending")
    .when(trim(lower(col("status"))).isin("invalid"), "Invalid")
    .when(trim(lower(col("status"))).isin("unknown", ""), "Unknown")
    .otherwise("Unknown")  # catch any other unexpected values
)

# Preview cleaned statuses
df_cleaned_kyc_documents.select("status").distinct().show(truncate=False)


StatementMeta(, 814c0d20-b57c-4f2a-a4db-167e55982b69, 20, Finished, Available, Finished)

+--------+
|  status|
+--------+
|Verified|
|Rejected|
| invalid|
| UNKNOWN|
| Pending|
|    NULL|
+--------+

+--------+
|status  |
+--------+
|Verified|
|Rejected|
|Unknown |
|Pending |
|Invalid |
+--------+



#### **Step 5: Data Transformation to verified_by column**
Issues
- The verified_by column had email addresses (verifiers) associated with incomplete or invalid statuses, such as:
- Pending, Invalid, Unknown

This created data integrity issues since documents that are not yet verified (or have failed verification) should not have a verifier recorded.

Transformations:
- Defined a list of non-verified statuses: ["Pending", "Invalid", "Unknown"].
- Applied a transformation where:
- If a document's status is in the above list, the corresponding verified_by value was set to NULL.
- Otherwise, the original verified_by value was retained.

In [19]:
# Define statuses that should NOT have a verifier
invalid_verification_statuses = ["Pending", "Invalid", "Unknown"]

# Clean and standardize the `verified_by` column:
# If the status is Pending, Invalid, or Unknown → set verified_by to NULL
df_cleaned_kyc_documents = df_cleaned_kyc_documents.withColumn(
    "verified_by",
    when(
        col("status").isin(invalid_verification_statuses),
        lit(None)
    ).otherwise(col("verified_by"))
)

# Show the cleaned result
df_cleaned_kyc_documents.show(20, truncate=False)


StatementMeta(, 814c0d20-b57c-4f2a-a4db-167e55982b69, 21, Finished, Available, Finished)

+------------------------------------+----------+-------------+-----------+-------+-----------+
|DOCUMENT_ID                         |USER_ID   |document_type|UPLOAD_DATE|status |verified_by|
+------------------------------------+----------+-------------+-----------+-------+-----------+
|f72dd9e6-4214-49b4-b008-7e031f8fd7d2|BB10000258|National ID  |2023-12-14 |Unknown|NULL       |
|424cf414-f9f1-4ef0-91e0-2dca665f8d2d|BB10000119|National ID  |2024-05-28 |Invalid|NULL       |
|42aea1e6-cd10-4304-92b0-02aced44091a|BB10000448|National ID  |2024-04-03 |Invalid|NULL       |
|535a0bc7-fe4f-4626-8d42-ee8d746f6e7c|BB10000565|National ID  |2023-10-20 |Unknown|NULL       |
|3528871f-20c4-4d4c-9376-2ef4eab2cee3|BB10001007|National ID  |2023-10-23 |Invalid|NULL       |
|32b4c075-91b6-4063-b88f-0dcb6f0892a9|BB10001957|National ID  |2023-07-11 |Invalid|NULL       |
|63b7af93-b182-4cf4-995c-ed1cbe1ef17a|BB10002936|National ID  |2023-11-22 |Invalid|NULL       |
|2feea3b5-a275-4dde-b8ae-114437f6d456|BB

#### **Step 6: Deduplication Based on user_id and Latest Upload**

Issue
- Multiple records existed for the same user_id, often with different values for fields like document_id, upload_date, or verified_by.
- This led to inconsistencies when joining or aggregating KYC information per user.

Transformation
- Partitioned the data by user_id and sorted by upload_date in descending order.
- Assigned a row number to each partition and retained only the most recent record (i.e., the one with row number 1).
- Dropped the temporary row number column used for ranking.

Outcome
- The dataset now contains only one record per user, representing the most recent and relevant KYC submission.

In [20]:
# Define deduplication window partitioned by user_id and ordered by upload_date descending
window_spec = Window.partitionBy("user_id").orderBy(col("upload_date").desc())

# Add row number to each partition to mark the latest record per user
df_deduplicated_kyc_documents = df_cleaned_kyc_documents.withColumn(
    "row_num", row_number().over(window_spec)
).filter(
    col("row_num") == 1
).drop("row_num")  # Drop the helper column

# Show result (optional)
df_deduplicated_kyc_documents.show(truncate=False)


StatementMeta(, 814c0d20-b57c-4f2a-a4db-167e55982b69, 22, Finished, Available, Finished)

+------------------------------------+----------+--------------+-----------+--------+---------------------------------------+
|DOCUMENT_ID                         |USER_ID   |document_type |UPLOAD_DATE|status  |verified_by                            |
+------------------------------------+----------+--------------+-----------+--------+---------------------------------------+
|4edfb5d8-17a9-49a7-95d6-1e15c9f930d5|BB10000005|Passport      |2024-03-23 |Rejected|emily.jones@boltbettingcompliance.com  |
|c8324c52-e464-49a5-b83b-43a862f589b2|BB10000010|Driver License|2023-12-25 |Rejected|NULL                                   |
|1189ae22-0d76-45ff-9b4f-f664804d0352|BB10000012|Driver License|2023-07-29 |Rejected|NULL                                   |
|8f9244d6-9a96-4d30-82a0-14759fbea630|BB10000014|Utility Bill  |2024-03-28 |Verified|kevin.moore@boltbettingcompliance.com  |
|673dbb8e-dd42-48a3-afec-02459d161bfc|BB10000019|National ID   |2023-12-22 |Verified|nathan.lewis@boltbettingcomplianc

#### **Step 7: Data Load to Silver Lakehouse Layer**
This script processes cleaned transaction data and writes it to the Silver layer in Delta format. It performs incremental loading using a MERGE strategy to avoid duplicates and ensure upserts (insert new records only).

Steps Performed:
Define Destination Path:
- Sets the location in the Lakehouse where the cleaned data will be saved:
- Files/cdp_silver/oracle/cleaned/cleaned_kyc_documents.

Prepare Columns for Consistency:
- upload_date: Ensures a proper date column is available for partitioning.
- load_timestamp: Adds a unique timestamp (once per job) for tracking the data load time.

Check if Delta Table Exists:
- If the table already exists at the path:
- Uses Delta Lake MERGE to compare existing and new data using the unique key document_id.
- Inserts only new (non-matching) records.

If the table does not exist:
- Creates the table with partitioning by upload_date for efficient querying and file organization.

In [21]:
# Define paths
silver_path = "Files/cdp_silver/oracle/cleaned/cleaned_kyc_documents"

# Add date column for partitioning (based on txn_timestamp if available)
df_cleaned_kyc_documents = df_cleaned_kyc_documents.withColumn(
    "upload_date", to_date("upload_date")  
)

# Add a consistent load timestamp column (once per job run)
load_ts = datetime.now().strftime("%Y%m%d%H%M%S")
df_cleaned_kyc_documents = df_cleaned_kyc_documents.withColumn(
    "load_timestamp", lit(load_ts)
)

# Perform MERGE if Delta table exists
if DeltaTable.isDeltaTable(spark, silver_path):
    delta_table = DeltaTable.forPath(spark, silver_path)

    delta_table.alias("target").merge(
        df_cleaned_kyc_documents.alias("source"),
        "target.document_id = source.document_id"  # Replace with your unique key
    ).whenNotMatchedInsertAll().execute()

else:
    # First write: create partitioned Delta table
    df_cleaned_kyc_documents.write.format("delta") \
        .mode("overwrite") \
        .partitionBy("upload_date") \
        .save(silver_path)


StatementMeta(, 814c0d20-b57c-4f2a-a4db-167e55982b69, 23, Finished, Available, Finished)